# D2 · What PCA sees in the digits: pixels, centroids and lost information

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Read a PCA on images: which pixels never vary, where the variance sits (center versus edge), how many components are needed for a given share of variance, how the classes arrange themselves in the reduced space (which digits get confused) and how much classification information
is lost by reducing to 2, 10 or 30 dimensions (with a centroid classifier).

## Dataset

`load_digits` (1797 8×8 images = 64 pixels, 10 classes): `X` (1797 × 64), `y`, and `images = X.reshape(-1, 8, 8)`.

## Educational purpose

Link the PCA numbers to the images: the pixels are also spatial coordinates, and the components are "images" (weights on the pixels). You will learn to ask the data "where is the variance" and to measure what a projection to 2 dimensions costs.

## Task

1. Save `constant_pixels` (number of pixels with zero variance) and `edge_variance_share`: share of the total variance of the pixels of the **outer ring** of the image (rows 0 and 7 and columns 0 and 7: 28 pixels) out of the 64.
2. Train `PCA().fit(X)`; save `k90` and `k99` (number of components for 90% and 99% of the cumulative variance) and `pc1_image` (the first component as an 8×8 matrix) with `pc1_center` = mean of |weight| of the 16 central pixels (rows 2-5, columns 2-5) minus the mean of the 28 edge pixels.
3. With the first two components compute the **centroids** of the 10 classes (`centroids`, 10 × 2 matrix) and the matrix of the distances between centroids; save `closest_pair` (the pair `(a, b)`, with `a < b`, of digits with the closest centroids) and `isolated_class` (the digit whose closest centroid is the farthest of all).
4. **Lost information:** with `NearestCentroid` in cross-validation (`StratifiedKFold(5, shuffle=True, random_state=0)`), on the components alone, compute `centroid_acc` (dictionary `k → accuracy`) for `k` in `[2, 10, 30, 64]`, with PCA inside the pipeline.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
images = X.reshape(-1, 8, 8)
edge = np.zeros((8, 8), dtype=bool)
edge[[0, 7], :] = True
edge[:, [0, 7]] = True
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
constant_pixels = None
edge_variance_share = None

In [ ]:
pca = PCA().fit(X)
k90 = None
k99 = None
pc1_image = None
pc1_center = None

In [ ]:
P2 = PCA(2).fit_transform(X)
centroids = None
closest_pair = None
isolated_class = None

In [ ]:
centroid_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Edge: `X.var(0).reshape(8, 8)[edge].sum() / X.var(0).sum()`. Centroids: `np.array([P2[y == c].mean(0) for c in range(10)])`; distances: `np.linalg.norm(centroids[:, None] - centroids[None], axis=2)` with the diagonal at infinity; pair: `np.unravel_index(D.argmin(), D.shape)`. Isolated: `D.min(axis=1).argmax()`. Center: `np.abs(pca.components_[0].reshape(8, 8))[2:6, 2:6].mean() - np.abs(...)[edge].mean()`.'):
    check(constant_pixels == (X.var(0) == 0).sum() and constant_pixels >= 3, "some pixels (at the corners) never vary", "(X.var(0) == 0).sum()")
    check(close(edge_variance_share, X.reshape(-1, 64).var(0)[edge.ravel()].sum() / X.var(0).sum()) and edge_variance_share < 0.25, "the outer ring (28 pixels out of 64) contains less than a quarter of the total variance", "sum of the variances of the 28 edge pixels / total")
    check(k90 == int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9)) + 1 and 18 <= k90 <= 24 and 38 <= k99 <= 48, "about 21 components for 90% and about 41 for 99% (out of 64 pixels)", "np.searchsorted(np.cumsum(explained_variance_ratio_), threshold) + 1")
    check(pc1_image.shape == (8, 8) and pc1_center > 0.05, "the first component weighs the central pixels more than the edge ones", "components_[0].reshape(8, 8)")
    check(centroids.shape == (10, 2) and close(centroids[3], P2[y == 3].mean(0)) and closest_pair[0] < closest_pair[1] and isolated_class == 0, "centroids computed; the digit 0 is the most isolated", "mean of the scores per class and distance matrix")
    check(set(centroid_acc) == {2, 10, 30, 64} and centroid_acc[2] < 0.65 and centroid_acc[10] > 0.85 and centroid_acc[64] > 0.88, "with 2 components a centroid classifier has less than 0.65; with 10 over 0.85; with 64 over 0.88", "NearestCentroid inside the pipeline with PCA(k)")
    check(centroid_acc[10] > centroid_acc[2] + 0.25, "going from 2 to 10 components gains over 25 points", "the 2D map loses a lot of class information")
done()

---
The commented solution is in `solutions/D_interpret/D2_what_pca_sees_in_the_digits.ipynb`. Open it after you pass the check, or after you have really tried.